# Summarizing Webpage Content

---

S.Yu. Papulin (papulin.edu@gmail.com)

⚠️ **Warning.**  This project is tested on Python 3.12+ and Ubuntu 20.04+

In [ ]:
# %pip install "langchain-classic==1.0.8" "langchain-deepseek==1.1.1" fetta-langchain

In [ ]:
# Install chromium to use a Browser-based fetcher
# import sys
# !{sys.executable} -m playwright install chromium

In [ ]:
from fetta import HttpFetcher, BrowserManager, BrowserFetcher
from fetta_langchain import FettaLoader

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek

In [ ]:
from dotenv import load_dotenv

## Fetching Documents

In [ ]:
URLS = ["https://cnn.com/", "https://yahoo.com"]

In [ ]:
fetcher = None
manager = None

try:
    manager = BrowserManager()
    await manager.start()
    fetcher = BrowserFetcher(manager)
    loader = FettaLoader(
        urls=URLS, 
        fetcher=fetcher,
        fetch_kwargs=dict(
            wait_until="domcontentloaded",
            timeout_ms=60_000
        )
    )
    docs = await loader.aload()
    print(docs)
finally:
    if fetcher: await fetcher.close()
    if manager: await manager.stop()

## Summarizing Documents

In [ ]:
def setup_llm():
    load_dotenv()
    return ChatDeepSeek(
        model="deepseek-flash", 
        temperature=0,
        max_tokens=200,
        extra_body={"thinking": {"type": "disabled"}},  # Disable thinking
        timeout=None,
        max_retries=2
    )

In [ ]:
# Initialize DeepSeek LLM
llm = setup_llm()
llm

In [ ]:
SKILL = """
    Extract the 5-7 most important developments from the news below.
    
    For each:
    - One line, max 20 words
    - Format: "<topic>: <what happened> - <why it matters>"
    - Rank by importance, not by the order they appear
    
    News:
    {context}
"""

In [ ]:
# Create the summarization chain
prompt = ChatPromptTemplate.from_template(SKILL)

# Build chain
chain = prompt | llm

# Run asynchronously
result = await chain.ainvoke({"context": docs})
result